# <font color="#F48E16">XAI aplicada al Deep Learning: Dataset sintético para Interpretabilidad Global de una RNN</font>

Material generado por <a href="https://www.linkedin.com/in/christian-oliva-moya-ingeniero/">Christian Oliva</a>. Cualquier duda, sugerencia o errata, no duden en contactar.

**Versión 1.0** - 29 de agosto de 2025

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

## <font color="#F48E16">Construcción del dataset sintético BxA</font>

Para construir el dataset sintético:

1. Generamos números aleatorios discretos para procesar correctamente la cadena, es decir, como si fuese un problema regular.

2. Obtenemos la salida de la cadena, que empiece por b=1 y termine por a=0.

3. Generamos el dataset con números reales, teniendo en cuenta que vamos a reemplazar cada a=0 por random_positivo y cada b=1 por random_negativo.

In [ ]:
def build_dataset(T=20, D=10000, C=2):
  dataset_int = np.random.randint(0, 2, (D, T)) # Si 0 = a, Si 1 = b
  dataset_y = (dataset_int[:, 0] == 1) & (dataset_int[:, -1] == 0)
  dataset_x = np.zeros((D, T, C))
  random_negativo = np.random.randn(D, T, C)*0.2 - 1.5
  random_positivo = np.random.randn(D, T, C)*0.2 + 1.5
  dataset_x[dataset_int == 0] = random_positivo[dataset_int == 0]
  dataset_x[dataset_int == 1] = random_negativo[dataset_int == 1]
  return dataset_int, dataset_x, dataset_y

In [ ]:
def transformar(cadena):
    mapeo = {0: 'a', 1: 'b'}
    return ''.join(mapeo[num] for num in cadena)

In [ ]:
dataset_int, dataset_x, dataset_y = build_dataset(D=10000)
dataset_int_test, dataset_x_test, dataset_y_test = build_dataset(D=5000)

In [ ]:
item=0
plt.figure(figsize=(10, 3))
plt.title(transformar(dataset_int[item]) + " - aceptar:" + str(dataset_y[item]))
plt.plot(dataset_x[item])
plt.show()

## <font color="#F48E16">Entrenamiento de una RNN "normal"</font>

Una red neuronal "normal" tiene tiene una capa de entrada, una capa recurrente y una capa densa de salida con activación o binaria o softmax. Uso softmax para generalizar, aunque podría usar binaria.

In [ ]:
def build_model_normal(input_shape, output_dim, units, optimizer="Adam"):
  input_layer = tf.keras.layers.Input(shape=input_shape)
  rec_layer = tf.keras.layers.SimpleRNN(units=units)(input_layer)
  output_layer = tf.keras.layers.Dense(output_dim, activation="softmax")(rec_layer)

  model = tf.keras.Model(inputs=input_layer, outputs=output_layer)
  model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])
  return model

Sin embargo, voy a utilizar la siguiente función porque necesito guardarme el `return_sequences=True`. Esta segunda función hace exactamente lo mismo que la anterior, pero permite devolver también `model_rec` para tener el vector $\mathbf{h}_t$ en cada instante de tiempo.

In [ ]:
def build_model(input_shape, output_dim, units, optimizer="Adam"):
  input_layer = tf.keras.layers.Input(shape=input_shape)
  # Comparando con la funcion build_model_normal, añado return_sequences=True y me quedo
  # con el último instante de tiempo (es lo mismo que return_sequences=False)
  # Así puedo recuperar todos los instantes de tiempo recurrentes de la red
  rec_layer = tf.keras.layers.SimpleRNN(units=units, return_sequences=True)(input_layer)
  last_timestep = rec_layer[:, -1, :]
  output_layer = tf.keras.layers.Dense(output_dim, activation="softmax")(last_timestep)

  model = tf.keras.Model(inputs=input_layer, outputs=output_layer)
  model_rec = tf.keras.Model(inputs=input_layer, outputs=rec_layer)
  model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])
  return model, model_rec

**Observación**: Es posible que esta red no alcance un 100% de acierto, dependiendo de la ejecución. No queremos quedarnos con eso, sino más bien ver el comportamiento de los estados internos.

In [ ]:
input_shape = dataset_x[0].shape
output_dim = len(np.unique(dataset_y))
units = 20

model_normal, model_rec = build_model(input_shape, output_dim, units)
model_normal.summary()

In [ ]:
model_normal.fit(dataset_x, dataset_y, validation_data=(dataset_x_test, dataset_y_test), epochs=10)

Obtenemos el espacio de estados interno de las 5 primeras neuronas como ejemplo:

In [ ]:
hs = model_rec.predict(dataset_x)
hs.shape

In [ ]:
for neurona in range(5):
  plt.figure(figsize=(10, 1))
  plt.plot(hs[:, :, neurona], '.', color="blue", markersize=1, alpha=0.1)
  plt.show()

Como hemos visto, la activación de las neuronas tiene cualquier valor real entre -1 y 1 (tanh) en cada instante de tiempo. Discretizar aquí puede ser complicado. ¿Cuántos estados distintos hay? En mi ejecución, 200.000 de 200.000, es decir, en todo el dataset no se ha repetido ningún estado en ningún momento (capacidad de generalización NULA).

In [ ]:
np.unique(np.concatenate(hs), axis=0).shape

**¿Observaciones?**

- Como hemos dicho, el espacio de estados interno de las redes neuronales recurrentes **es continuo**. Se ve porque cada neurona ocupa todo el rango (-1, 1) porque su función de activación es una tanh.

- La capacidad de generalización de la red es nula, por lo que no se debería poder extraer un conjunto de estados para generar el autómata.

# <font color="#F48E16">Construcción de un modelo interpretable</font>

## <font color="#F48E16"><b>Requisito 1</b>: Entrenamiento de una RNN con función de redondeo</font>

Primero hay que definir la función de redondeo, modificando su paso hacia atrás en el cálculo de la derivada para seguir utilizando la derivada de la tanh. Eso se hace con `@tf.custom_gradient` en Tensorflow.

In [ ]:
def round_tanh(decimals=0):
  @tf.custom_gradient
  def _rounded_tanh(x):
    scale = 10**decimals
    z = tf.tanh(x)
    output = tf.math.round(z * scale) / scale

    def backward(dy):
      custom_grad = 1.0 - tf.square(z)
      return dy * custom_grad
    return output, backward

  return _rounded_tanh

Ahora utilizamos esta función de activación en la capa recurrente. Además, añadimos regularización L1 para más adelante.

In [ ]:
def build_model_rnn_redondeada(input_shape, output_dim, units, wxh_l1reg=0.0, whh_l1reg=0.0, optimizer="Adam"):
  input_layer = tf.keras.layers.Input(shape=input_shape)
  # Comparando con la funcion build_model, añado la activación custom que acabo de crear
  # y añado regularización L1 tanto a Wxh como a Whh
  rec_layer = tf.keras.layers.SimpleRNN(units=units, return_sequences=True,
                                        activation=round_tanh(decimals=0),
                                        kernel_regularizer=tf.keras.regularizers.l1(wxh_l1reg),
                                        recurrent_regularizer=tf.keras.regularizers.l1(whh_l1reg))(input_layer)
  last_timestep = rec_layer[:, -1, :]
  output_layer = tf.keras.layers.Dense(output_dim, activation="softmax")(last_timestep)

  model = tf.keras.Model(inputs=input_layer, outputs=output_layer)
  model_rec = tf.keras.Model(inputs=input_layer, outputs=rec_layer)
  model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])
  return model, model_rec

**Observación**: No quiero que nos fijemos en si aprende o no porque el modelo está a medias aún, quiero que veáis cómo se comportan las neuronas ocultas.

In [ ]:
input_shape = dataset_x[0].shape
output_dim = len(np.unique(dataset_y))
units = 20
wxh_l1reg = 0.0
whh_l1reg = 0.0
learning_rate=1e-2
optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

model_normal, model_rec = build_model_rnn_redondeada(input_shape, output_dim, units,
                                                     optimizer=optimizer,
                                                     wxh_l1reg=wxh_l1reg,
                                                     whh_l1reg=whh_l1reg)
model_normal.summary()

In [ ]:
model_normal.fit(dataset_x, dataset_y, validation_data=(dataset_x_test, dataset_y_test), epochs=10)

In [ ]:
hs = model_rec.predict(dataset_x)
hs.shape

In [ ]:
for neurona in range(5):
  plt.figure(figsize=(10, 1))
  plt.plot(hs[:, :, neurona], '.', color="blue", markersize=1, alpha=0.1)
  plt.show()

In [ ]:
np.unique(np.concatenate(hs), axis=0).shape

**¿Observaciones?**

- Las neuronas de la capa oculta ahora sí están discretizadas gracias a la función de redondeo.

- De esos 200.000 estados posibles en entrenamiento, ahora solamente hay 660 diferentes, es decir, tenemos una buena capacidad de generalización.

- ¿Qué sucedería si añadimos regularización en la capa recurrente?

## <font color="#F48E16"><b>Requisito 2</b>: Entrenamiento de una RNN con capa discretizadora</font>

El siguiente paso es añadir la capa discretizadora para definir los símbolos del autómata, ya que si queremos interpretar la RNN como un autómata, necesitamos entradas discretas.

Para ello, añadimos una capa densa pre-recurrente con activación también redondeada (podría ser sigmoid, relu, pero hemos optado por mantener la tanh) y regularización L1.

In [ ]:
def build_model_rnn_discretizadora(input_shape, output_dim, discretizer_units, units, wxh_l1reg=0.0, whh_l1reg=0.0, discretizer_l1reg=0.0, optimizer="Adam"):
  input_layer = tf.keras.layers.Input(shape=input_shape)
  # Comparando con build_model_rnn_redondeada, añado una capa densa adicional, la capa discretizadora,
  # con la función de activación redondeada
  discretizer_layer = tf.keras.layers.Dense(discretizer_units, activation=round_tanh(decimals=0),
                                            kernel_regularizer=tf.keras.regularizers.l1(discretizer_l1reg))(input_layer)
  rec_layer = tf.keras.layers.SimpleRNN(units=units, return_sequences=True,
                                        activation=round_tanh(decimals=0),
                                        kernel_regularizer=tf.keras.regularizers.l1(wxh_l1reg),
                                        recurrent_regularizer=tf.keras.regularizers.l1(whh_l1reg))(discretizer_layer)
  last_timestep = rec_layer[:, -1, :]
  output_layer = tf.keras.layers.Dense(output_dim, activation="softmax")(last_timestep)

  model = tf.keras.Model(inputs=input_layer, outputs=output_layer)
  model_rec = tf.keras.Model(inputs=input_layer, outputs=rec_layer)
  # Añado también model_discretizer para ver la transformación de continuo a discreto
  model_discretizer = tf.keras.Model(inputs=input_layer, outputs=discretizer_layer)
  model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])
  return model, model_rec, model_discretizer

**Observación**: No quiero que nos fijemos en si aprende o no porque el modelo está a medias aún, quiero que veáis cómo se comporta la capa discretizadora.

In [ ]:
input_shape = dataset_x[0].shape
output_dim = len(np.unique(dataset_y))
units = 20
discretizer_units = 3
wxh_l1reg = 0.0
whh_l1reg = 0.0
discretizer_l1reg = 0.0
learning_rate=1e-2
optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

model_normal, model_rec, model_discretizer = build_model_rnn_discretizadora(input_shape, output_dim,
                                                                            discretizer_units, units,
                                                                            optimizer=optimizer,
                                                                            wxh_l1reg=wxh_l1reg,
                                                                            whh_l1reg=whh_l1reg,
                                                                            discretizer_l1reg=discretizer_l1reg)
model_normal.summary()

In [ ]:
model_normal.fit(dataset_x, dataset_y, validation_data=(dataset_x_test, dataset_y_test), epochs=10)

In [ ]:
hs = model_rec.predict(dataset_x)
hs.shape

In [ ]:
xs = model_discretizer.predict(dataset_x)
xs.shape

In [ ]:
for neurona in range(5):
  plt.figure(figsize=(10, 1))
  plt.plot(hs[:, :, neurona], '.', color="blue", markersize=1, alpha=0.1)
  plt.ylim(-1.05, 1.05)
  plt.show()

In [ ]:
np.unique(np.concatenate(hs), axis=0).shape

Vamos a dibujar los dos canales de entrada, que no lo hemos hecho antes. Como podemos ver, se diferencian dos nubes de puntos (tal y como hemos definido al construir el dataset). Recordad que los valores positivos en ambos canales representa el símbolo `a` y los valores negativos el símbolo `b`.

In [ ]:
plt.plot(np.concatenate(dataset_x)[:, 0], np.concatenate(dataset_x)[:, 1], '.')
plt.show()

Vamos a ver qué está haciendo la capa discretizadora.

In [ ]:
np.unique(np.concatenate(xs), axis=0).shape

Primero me guardo un par de diccionarios para traducir vector de la capa discretizadora a token (símbolo discreto).

In [ ]:
xs2simbolo = {}
simbolo2xs = {}
for c, vector in enumerate(np.unique(np.concatenate(xs), axis=0)):
  vector = tuple([int(i) for i in vector])
  xs2simbolo[vector] = c
  simbolo2xs[c] = vector

In [ ]:
xs2simbolo

Con el siguiente código, vamos a generar una rejilla donde explorar todo el espacio posible de entrada.

In [ ]:
# Definir el tamaño de la rejilla y los límites
grid_size = 100
x_min, x_max = -3, 3
y_min, y_max = -3, 3

# Crear los puntos de la rejilla
x = np.linspace(x_min, x_max, grid_size)
y = np.linspace(y_min, y_max, grid_size)
X, Y = np.meshgrid(x, y)

# Aplanar para obtener todas las combinaciones (10000 puntos si es 100x100)
X_flat = X.ravel()
Y_flat = Y.ravel()
puntos = np.column_stack((X_flat, Y_flat))  # Convertir a pares (x, y)
generado = np.zeros((len(puntos)))

from tqdm import tqdm
for i in tqdm(range(len(puntos))):
    vector = np.zeros((2))
    vector[0] = puntos[i][0]
    vector[1] = puntos[i][1]
    discr = model_discretizer(np.repeat(vector[None, None, :], 20, axis=1)).numpy()[0][0]
    try:
      discr = tuple([int(i) for i in discr])
      generado[i] = xs2simbolo[discr]
    except:
      generado[i] = -1

Lo pintamos para que veáis el resultado.

In [ ]:
Z_flat = generado  # Evaluar los puntos en el modelo
Z = Z_flat.reshape(grid_size, grid_size)  # Volver a darle forma para visualización

# Graficar la evaluación del modelo en el espacio 2D
plt.figure(figsize=(8, 6))
plt.contourf(X, Y, Z, levels=50, alpha=0.7, cmap="jet")  # Mapa de calor de las salidas
plt.colorbar(label="Salida del modelo")
plt.xlabel("Atributo 1")
plt.ylabel("Atributo 2")
plt.grid(alpha=0.5)
plt.plot(np.concatenate(dataset_x)[:, 0], np.concatenate(dataset_x)[:, 1], '.', color="magenta")
plt.show()

**¿Observaciones?**

- El espacio de entrada continuo se discretiza según una serie de hiperplanos. Concretamente **dos hiperplanos paralelos** por cada neurona activa de la capa discretizadora. Como hay 3 neuronas activas, hay 3 pares de hiperplanos.

- Hay puntos en el espacio que no están definidos (son situaciones que no pueden surgir en la naturaleza del problema según los datos de entrenamiento).

## <font color="#F48E16"><b>Requisito 3</b>: Entrenamiento de una RNN con capa post-recurrente</font>

Ahora sí, añadiendo este último requisito, **tendremos el modelo completamente interpretable**. Para ello, añadimos la capa post-recurrente. Simplemente, después de la capa recurrente, añadimos una densa con una única neurona y activación sigmoide (para tener $y_t = d(\mathbf{h}_t)$ tal y como hemos descrito previamente). Luego concatenamos instantes de tiempo con una capa Flatten y, a partir de ahí, todo normal.

In [ ]:
def build_model_rnn_interpretable(input_shape, output_dim, discretizer_units, units,
                                  wxh_l1reg=0.0, whh_l1reg=0.0, discretizer_l1reg=0.0,
                                  prm_l1reg=0.0, optimizer="Adam"):
  input_layer = tf.keras.layers.Input(shape=input_shape)
  # Comparando con build_model_rnn_redondeada, añado una capa densa adicional, la capa discretizadora,
  # con la función de activación redondeada
  discretizer_layer = tf.keras.layers.Dense(discretizer_units, activation=round_tanh(decimals=0),
                                            kernel_regularizer=tf.keras.regularizers.l1(discretizer_l1reg))(input_layer)
  rec_layer = tf.keras.layers.SimpleRNN(units=units, return_sequences=True,
                                        activation=round_tanh(decimals=0),
                                        kernel_regularizer=tf.keras.regularizers.l1(wxh_l1reg),
                                        recurrent_regularizer=tf.keras.regularizers.l1(whh_l1reg))(discretizer_layer)

  # Comparando con build_model_rnn_discretizadora, añado una capa densa d(ht) con 1 neurona y activacion sigmoid
  prm_layer = tf.keras.layers.Dense(1, activation="sigmoid")(rec_layer)
  # Luego, necesito hacer un flatten temporal para tener un vector y[1::T]
  flatten_layer = tf.keras.layers.Flatten()(prm_layer)
  # Por último, proceso cada instante de tiempo. Añado regularización para que se anulen tiempos innecesarios (si esque los hay)
  output_layer = tf.keras.layers.Dense(output_dim, activation="softmax",
                                       kernel_regularizer=tf.keras.regularizers.l1(prm_l1reg))(flatten_layer)

  model = tf.keras.Model(inputs=input_layer, outputs=output_layer)
  model_rec = tf.keras.Model(inputs=input_layer, outputs=rec_layer)

  model_discretizer = tf.keras.Model(inputs=input_layer, outputs=discretizer_layer)
  # Añado también model_prm para ver las yt = d(ht)
  model_prm = tf.keras.Model(inputs=input_layer, outputs=prm_layer)
  model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])
  return model, model_rec, model_discretizer, model_prm

Voy a apurar las dimensiones para que se vea lo más claro posible. Además voy a añadir regularización en cada capa.

In [ ]:
input_shape = dataset_x[0].shape
output_dim = len(np.unique(dataset_y))
units = 3
discretizer_units = 1
discretizer_l1reg = 1e-4
wxh_l1reg = 1e-4
whh_l1reg = 1e-4
prm_l1reg = 1e-4
learning_rate = 1e-2
optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

(model_normal,
 model_rec,
 model_discretizer,
 model_prm) = build_model_rnn_interpretable(input_shape, output_dim,
                                            discretizer_units, units,
                                            optimizer=optimizer,
                                            wxh_l1reg=wxh_l1reg,
                                            whh_l1reg=whh_l1reg,
                                            discretizer_l1reg=discretizer_l1reg,
                                            prm_l1reg=prm_l1reg)
model_normal.summary()

In [ ]:
model_normal.fit(dataset_x, dataset_y, validation_data=(dataset_x_test, dataset_y_test), epochs=10)

In [ ]:
hs = model_rec.predict(dataset_x)
hs.shape

In [ ]:
hs_all = np.concatenate(hs)
hs_all.shape

In [ ]:
np.unique(hs_all, axis=0).shape

In [ ]:
xs = model_discretizer.predict(dataset_x)
xs.shape

In [ ]:
xs_all = np.concatenate(xs)
xs_all.shape

In [ ]:
np.unique(xs_all, axis=0).shape

In [ ]:
ys = model_prm.predict(dataset_x)
ys.shape

In [ ]:
ys_all = np.concatenate(ys)
ys_all.shape

In [ ]:
np.unique(ys_all, axis=0).shape

### Visualización de la capa discretizadora

Hacemos lo mismo que antes, en este caso la capa discretizadora tiene una única neurona, por lo que se formarán **dos hiperplanos** que separan el espacio en **tres regiones**.

In [ ]:
xs2simbolo = {}
simbolo2xs = {}
for c, vector in enumerate(np.unique(np.concatenate(xs), axis=0)):
  vector = tuple([int(i) for i in vector])
  xs2simbolo[vector] = c
  simbolo2xs[c] = vector

In [ ]:
# Definir el tamaño de la rejilla y los límites
grid_size = 100
x_min, x_max = -3, 3
y_min, y_max = -3, 3

# Crear los puntos de la rejilla
x = np.linspace(x_min, x_max, grid_size)
y = np.linspace(y_min, y_max, grid_size)
X, Y = np.meshgrid(x, y)

# Aplanar para obtener todas las combinaciones (10000 puntos si es 100x100)
X_flat = X.ravel()
Y_flat = Y.ravel()
puntos = np.column_stack((X_flat, Y_flat))  # Convertir a pares (x, y)
generado = np.zeros((len(puntos)))

from tqdm import tqdm
for i in tqdm(range(len(puntos))):
    vector = np.zeros((2))
    vector[0] = puntos[i][0]
    vector[1] = puntos[i][1]
    discr = model_discretizer(np.repeat(vector[None, None, :], 20, axis=1)).numpy()[0][0]
    try:
      discr = tuple([int(i) for i in discr])
      generado[i] = xs2simbolo[discr]
    except:
      generado[i] = -1

Además, aprovecho para guardarme el traductor de token a símbolo y viceversa, que lo utilizaré para generar el autómata.

In [ ]:
traductor_simbolos2id = {}
traductor_id2simbolos = {}
va = model_discretizer.predict(np.repeat(np.array([1.5, 1.5])[None, None, :], 20, axis=1))[0, 0]
va = xs2simbolo[tuple([int(i) for i in va])]
traductor_simbolos2id["a"] = va
traductor_id2simbolos[va] = "a"
vb = model_discretizer.predict(np.repeat(np.array([-1.5, -1.5])[None, None, :], 20, axis=1))[0, 0]
vb = xs2simbolo[tuple([int(i) for i in vb])]
traductor_simbolos2id["b"] = vb
traductor_id2simbolos[vb] = "b"


Z_flat = generado  # Evaluar los puntos en el modelo
Z = Z_flat.reshape(grid_size, grid_size)  # Volver a darle forma para visualización

# Graficar la evaluación del modelo en el espacio 2D
plt.figure(figsize=(8, 6))
plt.contourf(X, Y, Z, levels=50, alpha=0.7, cmap="jet")  # Mapa de calor de las salidas
plt.colorbar(label="Salida del modelo")
plt.xlabel("Atributo 1")
plt.ylabel("Atributo 2")
plt.grid(alpha=0.5)
plt.plot(np.concatenate(dataset_x)[:, 0], np.concatenate(dataset_x)[:, 1], '.', color="magenta")
plt.annotate("$\\hat{\\mathbf{x}}^"+str(traductor_simbolos2id["a"])+"$", (1.5, 1.5), size=18)
plt.annotate("$\\hat{\\mathbf{x}}^"+str(traductor_simbolos2id["b"])+"$", (-1.5, -1.5), size=18)
plt.show()

### Construcción del autómata incluyendo con variable acumulativa

Es la primera vez que mostramos una función para construir un autómata. Vamos a describirla poco a poco.

In [ ]:
from graphviz import Digraph
from matplotlib.colors import LinearSegmentedColormap

def crear_diagrama_de_estados(estados, transiciones, estado_inicial, z_values, traductor_simbolos=None):
    fsm = Digraph(format='png')
    fsm.attr(rankdir='LR')

    fsm.node("", shape="none", label="")  # Nodo invisible para la flecha inicial
    fsm.edge("", str(estado_inicial))  # Flecha hacia el estado inicial

    # Crear un colormap personalizado de verde (#00FF00) a blanco (#FFFFFF)
    custom_cmap = LinearSegmentedColormap.from_list("custom_green_white", ["#FFFFFF", "#00FF00"])

    # Función para generar el color de cada nodo basado en z(h)
    def get_color(z):
        rgba = custom_cmap(z[0])  # z debe estar entre 0 y 1
        r, g, b, _ = [int(255 * v) for v in rgba]
        return f"#{r:02x}{g:02x}{b:02x}"

    # Añadir los estados (nodos)
    for estado in estados:
        if str(estado) == "-2":
            continue
        # Asignar un color según el valor de z(h)
        color = get_color(z_values.get(estado, 0))  # Si no hay z(h), asigna 0 (vacío)
        fsm.node(str(estado), label=str(estado)+"\ny="+str(np.around(z_values.get(estado, 0)[0], decimals=2)),
                 color="black", fillcolor=color, style="filled", fontcolor="black", shape="ellipse", width="0.6")

    # Añadir las transiciones (aristas)
    for origen in transiciones:
        if str(origen) == "-2":
            continue
        for simbolo in transiciones[origen]:
            destino = transiciones[origen][simbolo]
            if str(destino) == "-2":
                continue
            if traductor_simbolos is not None:
                label = str(traductor_simbolos[simbolo])
            else:
                label = str(simbolo)
            fsm.edge(str(origen), str(destino), label=label)

    return fsm

Necesito un traductor de input $\mathbf{x}_t$ a token discreto.

In [ ]:
x2id = {}
id2x = {}
for i, x in enumerate(np.unique(xs_all, axis=0).astype(int)):
    x2id[tuple(x)] = i
    id2x[i] = tuple(x)

In [ ]:
xs_token = []
for x in xs:
    x_token = []
    for xt in x:
        token = tuple(xt.astype(int))
        x_token.append(x2id[token])
    xs_token.append(x_token)
xs_token = np.array(xs_token)
xs_token.shape

También necesito un traductor de estado interno $\mathbf{h}_t$ a token de estado.

In [ ]:
h2id = {}
id2h = {}
for i, h in enumerate(np.unique(hs_all, axis=0).astype(int)):
    h2id[tuple(h)] = i
    id2h[i] = tuple(h)

Aquí aprovecho para guardarme el valor $d(\mathbf{h}_t)$ asociado, lo voy a guardar en `h2z`.

In [ ]:
hs_token = []
h2z = {}
for h, z in zip(hs, ys):
    h_token = []
    for ht, zt in zip(h, z):
        token = tuple(ht.astype(int))
        h_token.append(h2id[token])
        if token not in h2z:
            h2z[h2id[token]] = zt
        else:
            if h2z[h2id[token]] != zt:
                print(" ALGO VA MAL")
    hs_token.append(h_token)
hs_token = np.array(hs_token)
hs_token.shape

In [ ]:
h2z

Genero la tabla de transiciones del autómata mirando las transiciones entre estados en cada instante de tiempo para cada cadena de entrenamiento. Si hubiese una transición no-determinista, se avisa por terminal.

In [ ]:
from tqdm import tqdm
tabla_transiciones = {}
for i in tqdm(range(len(hs_token))):
    for t in range(hs_token.shape[1]):
        simbolo = xs_token[i, t]
        if t == 0:
            estado_ini = -1
        else:
            estado_ini = hs_token[i, t-1]

        estado_fin = hs_token[i, t]

        if estado_ini in tabla_transiciones:
            if simbolo in tabla_transiciones[estado_ini]:
                if estado_fin != tabla_transiciones[estado_ini][simbolo]:
                    print(" > ALGO VA MAL")
                    print("    tiempo:", t)
                    print("       ini:", estado_ini)
                    print("   simbolo:", simbolo)
                    print("       fin:", estado_fin)
                    print("     trans:", tabla_transiciones[estado_ini][simbolo])
                else:
                    tabla_transiciones[estado_ini][simbolo] = estado_fin
            else:
                tabla_transiciones[estado_ini][simbolo] = estado_fin
        else:
            tabla_transiciones[estado_ini] = {simbolo: estado_fin}

In [ ]:
# tabla_transiciones[estado_ini][simbolo] ---> estado_fin

Por fin, llamamos a la función inicial para crear el autómata finito con variable acumulativa. Aquí le paso el traductor para que ponga los nombres reales de los símbolos en lugar de los tokens. La $y$ está redondeada a 2 decimales.

In [ ]:
crear_diagrama_de_estados(list(id2h.keys()), tabla_transiciones, -1, h2z, traductor_simbolos=traductor_id2simbolos)

Junto al autómata de arriba, necesitamos ver las ponderaciones temporales de $\mathbf{W}^p$ para entender qué está pasando.

In [ ]:
seq_len = dataset_x.shape[1]
plt.figure(figsize=(15, 2))
plt.bar(range(seq_len), model_normal.layers[-1].weights[0].numpy()[:, -1])
plt.grid()
plt.xticks(range(seq_len), range(1, seq_len+1))
plt.xlabel("Tiempo")
plt.ylabel("wp")
plt.show()

**Preguntas:**

- ¿Somos capaces de entender el autómata?

- ¿Podemos afirmar que este autómata es directamente equivalente a la red neuronal?

**Vamos a probar los cuatro escenarios posibles con el siguiente código**

In [ ]:
cadena1 = np.ones((20, 2))
cadena2 = np.ones((20, 2))
cadena2[0] *= -1
cadena3 = np.ones((20, 2))
cadena3[-1] *= -1
cadena4 = np.ones((20, 2))
cadena4[0] *= -1
cadena4[-1] *= -1

cadenas = [cadena1, cadena2, cadena3, cadena4]

# A PARTIR DE AQUI SE ENCARGA DE PINTAR

for cadena in cadenas:

  z_values = (model_prm.predict(cadena[None, :, :])[0, :, 0] * model_normal.layers[-1].weights[0].numpy()[:, -1]).cumsum()
  print(z_values)
  h_values =  []
  h = model_rec.predict(cadena[None, :, :])[0]
  for ht in h:
      token = tuple(ht.astype(int))
      h_values.append(h2id[token])

  x_values = []
  x = model_discretizer.predict(cadena[None, :, :])[0]
  for xt in x:
      token = tuple(xt.astype(int))
      x_values.append(x2id[token])

  def get_z_scaled(z, minimum=-14, maximum=14):
      z_clipped = np.clip(z, minimum, maximum)
      return (z_clipped+maximum)/(maximum-minimum)

  import matplotlib.patches as mpatch

  fig, ax = plt.subplots(figsize=(20,1))

  z_scaled = get_z_scaled(0.0)
  colors = [
      (0.0, "#0000FF"),    # Azul en el inicio
      (0.5, "white"),   # Blanco en el medio
      (1.0, "#FF0000"),     # Rojo al final
  ]

  # Crear el colormap
  custom_cmap = LinearSegmentedColormap.from_list("mi_colormap", colors)
  rgba = custom_cmap(z_scaled)
  r, g, b, _ = [int(255 * v) for v in rgba]
  color = f"#{r:02x}{g:02x}{b:02x}"

  rectangle = mpatch.Rectangle((0,0), 1, 1, color=color)
  ax.add_artist(rectangle)
  rx, ry = rectangle.get_xy()
  cx = rx + rectangle.get_width()/2.0
  cy = ry + rectangle.get_height()/2.0
  ax.annotate("-1", (cx, cy), color='k', weight='bold',
                  fontsize=14, ha='center', va='center')


  for t, (simbolo, z) in enumerate(zip(x_values, z_values)):
      z_scaled = get_z_scaled(z)
      rgba = custom_cmap(z_scaled)
      r, g, b, _ = [int(255 * v) for v in rgba]
      color = f"#{r:02x}{g:02x}{b:02x}"

      rectangle = mpatch.Rectangle((t+1,0), 1, 1, color=color)
      ax.add_artist(rectangle)
      rx, ry = rectangle.get_xy()
      cx = rx + rectangle.get_width()/2.0
      cy = ry + rectangle.get_height()/2.0

      ax.annotate(traductor_id2simbolos[simbolo], (cx, cy), color='k', weight='bold', # OJO AQUI a y b
                  fontsize=14, ha='center', va='center')

  rectangle = mpatch.Rectangle((t+3,0), 1, 1, color="#FFEE00" if (cadena == cadena2).all() else "#880088")
  ax.add_artist(rectangle)
  rx, ry = rectangle.get_xy()
  cx = rx + rectangle.get_width()/2.0
  cy = ry + rectangle.get_height()/2.0

  ax.annotate("T" if (cadena == cadena2).all() else "NT", (cx, cy), color='black' if (cadena == cadena2).all() else "white", weight='bold',
              fontsize=14, ha='center', va='center')

  ax.set_aspect('equal')
  ax.set_xlim((0, t+4))
  #plt.xticks([])
  ax.set_xticks(range(1, seq_len+1), range(1, seq_len+1))
  plt.yticks([])
  plt.ylabel("St", rotation="vertical", size=16)

  plt.show()

In [ ]:
z_values.shape